# Exp 66: Train From Prepared CSV

Colab-ready notebook для обучения `exp 66` только из `exp66_training_dataset.csv`.

In [ ]:
!pip install -q lightgbm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# Поправь путь под свой Google Drive
DATASET_PATH = Path('/content/drive/MyDrive/exp66_training_dataset.csv')
OUTPUT_DIR = Path('/content/drive/MyDrive/exp66_colab_outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATE_COL = 'Дата'
BAKERY_COL = 'Пекарня'
PRODUCT_COL = 'Номенклатура'
CATEGORY_COL = 'Категория'
DEMAND_TARGET = 'Спрос'

MODEL_PARAMS = {
    'n_estimators': 2304,
    'learning_rate': 0.016085231060110994,
    'num_leaves': 151,
    'max_depth': 7,
    'min_child_samples': 5,
    'subsample': 0.8012777641245349,
    'colsample_bytree': 0.6654847541174173,
    'reg_alpha': 6.633500153052146,
    'reg_lambda': 2.204771489304501e-06,
    'random_state': 42,
    'n_jobs': -1,
    'verbose': -1,
}

FEATURES_V2 = [
    'Пекарня', 'Номенклатура', 'Категория', 'Город',
    'ДеньНедели', 'День', 'IsWeekend', 'Месяц', 'НомерНедели',
    'sales_lag1', 'sales_lag2', 'sales_lag3', 'sales_lag7',
    'sales_lag14', 'sales_lag30',
    'sales_roll_mean3', 'sales_roll_mean7', 'sales_roll_std7',
    'sales_roll_mean14', 'sales_roll_mean30',
    'is_holiday', 'is_pre_holiday', 'is_post_holiday', 'is_payday_week',
    'is_month_start', 'is_month_end',
    'temp_max', 'temp_min', 'temp_mean', 'temp_range',
    'precipitation', 'rain', 'snowfall', 'windspeed_max',
    'is_rainy', 'is_snowy', 'is_cold', 'is_warm',
    'is_windy', 'is_bad_weather', 'weather_cat_code',
    'avg_price', 'price_vs_median', 'price_lag7',
    'price_change_7d', 'price_roll_mean7', 'price_roll_std7',
]

DEMAND_FEATURES = [
    'demand_lag1', 'demand_lag2', 'demand_lag3', 'demand_lag7',
    'demand_lag14', 'demand_lag30',
    'demand_roll_mean3', 'demand_roll_mean7', 'demand_roll_mean14', 'demand_roll_mean30',
    'demand_roll_std7',
]

EXP63_EXTRA_FEATURES = [
    'is_censored_lag1', 'lost_qty_lag1', 'pct_censored_7d',
    'sales_dow_mean', 'demand_dow_mean', 'demand_trend', 'cv_7d',
    'stale_ratio_lag1', 'items_in_bakery_today', 'items_in_bakery_lag1', 'items_change',
]

FEATURES_V3 = FEATURES_V2 + DEMAND_FEATURES
BASE_FEATURES = FEATURES_V3 + EXP63_EXTRA_FEATURES
CATEGORICAL_COLS = ['Пекарня', 'Номенклатура', 'Категория', 'Город', 'Месяц', 'cluster_loc', 'cluster_ts']

BASELINE_V3_MAE = 2.8816
EXP63_MAE = 2.8540

def wmape(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return np.sum(np.abs(y_true - y_pred)) / (np.sum(y_true) + 1e-8) * 100

def train_quantile(X_train, y_train, alpha=0.5, params=None):
    p = (params or MODEL_PARAMS).copy()
    p['objective'] = 'quantile'
    p['alpha'] = alpha
    p.pop('metric', None)
    model = LGBMRegressor(**p)
    model.fit(X_train, y_train)
    return model

def predict_clipped(model, X):
    return np.maximum(model.predict(X), 0)

def evaluate_model(name, train_df, test_df, features):
    available = [f for f in features if f in train_df.columns]
    model = train_quantile(train_df[available], train_df[DEMAND_TARGET], alpha=0.5)
    pred = predict_clipped(model, test_df[available])
    y_test = test_df[DEMAND_TARGET].to_numpy()
    mae = mean_absolute_error(y_test, pred)
    wm = wmape(y_test, pred)
    bias = float(np.mean(y_test - pred))
    r2 = r2_score(y_test, pred)
    result = {
        'mae': round(float(mae), 4),
        'wmape': round(float(wm), 2),
        'bias': round(float(bias), 4),
        'r2': round(float(r2), 4),
        'n_features': len(available),
        'delta_vs_v3': round(float(mae - BASELINE_V3_MAE), 4),
        'delta_vs_exp63': round(float(mae - EXP63_MAE), 4),
    }
    return model, pred, result, available

def evaluate_routed_model(train_df, test_df, features):
    available = [f for f in features if f in train_df.columns]
    preds = np.full(len(test_df), np.nan, dtype=float)
    cluster_results = []
    for cluster_id in sorted(train_df['cluster_ts'].astype(int).unique()):
        cluster_train = train_df[train_df['cluster_ts'].astype(int) == cluster_id]
        cluster_test = test_df[test_df['cluster_ts'].astype(int) == cluster_id]
        if cluster_test.empty or len(cluster_train) < 500:
            continue
        model = train_quantile(cluster_train[available], cluster_train[DEMAND_TARGET], alpha=0.5)
        cluster_pred = predict_clipped(model, cluster_test[available])
        preds[test_df['cluster_ts'].astype(int) == cluster_id] = cluster_pred
        cluster_results.append({
            'cluster_ts': int(cluster_id),
            'train_rows': int(len(cluster_train)),
            'test_rows': int(len(cluster_test)),
            'mae': round(float(mean_absolute_error(cluster_test[DEMAND_TARGET], cluster_pred)), 4),
        })
    missing_mask = np.isnan(preds)
    if missing_mask.any():
        fallback = train_quantile(train_df[available], train_df[DEMAND_TARGET], alpha=0.5)
        preds[missing_mask] = predict_clipped(fallback, test_df.loc[missing_mask, available])
    y_test = test_df[DEMAND_TARGET].to_numpy()
    mae = mean_absolute_error(y_test, preds)
    wm = wmape(y_test, preds)
    bias = float(np.mean(y_test - preds))
    r2 = r2_score(y_test, preds)
    result = {
        'mae': round(float(mae), 4),
        'wmape': round(float(wm), 2),
        'bias': round(float(bias), 4),
        'r2': round(float(r2), 4),
        'n_features': len(available),
        'delta_vs_v3': round(float(mae - BASELINE_V3_MAE), 4),
        'delta_vs_exp63': round(float(mae - EXP63_MAE), 4),
        'routed_clusters': cluster_results,
    }
    return preds, result, available

In [ ]:
df = pd.read_csv(DATASET_PATH, encoding='utf-8-sig')
df[DATE_COL] = pd.to_datetime(df[DATE_COL])

for col in CATEGORICAL_COLS:
    if col in df.columns:
        df[col] = df[col].astype('category')

train_df = df[df['split'] == 'train'].copy()
test_df = df[df['split'] == 'test'].copy()

print(df.shape)
print('train_rows =', len(train_df))
print('test_rows  =', len(test_df))
print('date range =', df[DATE_COL].min().date(), '--', df[DATE_COL].max().date())

In [ ]:
variants = {
    'A_exp63_baseline': BASE_FEATURES,
    'B_plus_cluster_loc': BASE_FEATURES + ['cluster_loc'],
    'C_plus_cluster_ts': BASE_FEATURES + ['cluster_ts'],
    'D_plus_both_clusters': BASE_FEATURES + ['cluster_loc', 'cluster_ts'],
}

all_results = {}
all_predictions = {}

for name, feats in variants.items():
    model, pred, result, available = evaluate_model(name, train_df, test_df, feats)
    all_results[name] = result
    all_predictions[name] = pred
    print(name, result)

pred_e, result_e, available_e = evaluate_routed_model(train_df, test_df, BASE_FEATURES + ['cluster_loc', 'cluster_ts'])
all_results['E_routed_cluster_ts'] = result_e
all_predictions['E_routed_cluster_ts'] = pred_e
print('E_routed_cluster_ts', result_e)

In [ ]:
summary = pd.DataFrame(all_results).T.sort_values('mae')
summary

In [ ]:
best_variant = summary.index[0]

predictions = test_df[[DATE_COL, BAKERY_COL, PRODUCT_COL, CATEGORY_COL, DEMAND_TARGET, 'cluster_loc', 'cluster_ts']].copy()
predictions = predictions.rename(columns={DEMAND_TARGET: 'fact_demand'})
for name, pred in all_predictions.items():
    predictions[f'pred_{name}'] = pred

metrics = {
    'experiment': '66_cluster_features_from_prepared_csv',
    'target': DEMAND_TARGET,
    'baseline_v3_mae': BASELINE_V3_MAE,
    'exp63_mae': EXP63_MAE,
    'best_variant': best_variant,
    'results': all_results,
    'train_rows': int(len(train_df)),
    'test_rows': int(len(test_df)),
}

metrics_path = OUTPUT_DIR / 'metrics.json'
preds_path = OUTPUT_DIR / 'predictions.csv'

with open(metrics_path, 'w', encoding='utf-8') as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)
predictions.to_csv(preds_path, index=False, encoding='utf-8-sig')

print(metrics_path)
print(preds_path)
print('best_variant =', best_variant)